# DeLM-Style Decentralized Multi-Agent System
## Shared Verified Context + Task Queue (Hugging Face Inference + LangGraph)

Inspired by **Decentralized Language Models (DeLM)** — arXiv:2606.10662

Instead of a central orchestrator bottleneck:
- Agents asynchronously claim tasks from a **task queue**
- They read a **shared verified context** of compact gists
- They write back only **verified** updates
- No single agent owns the merge step

Demo: collaborative codebase improvement (Analyzer / Improver / Reviewer roles that claim work themselves).

In [ ]:
import os
import json
import re
from typing import TypedDict, List, Optional
from dataclasses import dataclass, field
from huggingface_hub import InferenceClient
from langgraph.graph import StateGraph, END

HF_TOKEN = os.getenv("HF_TOKEN", "hf_your_token_here")
MODEL = "Qwen/Qwen2.5-7B-Instruct"

client = InferenceClient(token=HF_TOKEN)
print(f"DeLM-style system ready — model: {MODEL}")

In [ ]:
@dataclass
class Gist:
    """Compact verified update written to the shared context."""
    agent: str
    content: str
    verified: bool = False
    evidence: str = ""

class SharedContext:
    def __init__(self):
        self.gists: List[Gist] = []

    def add(self, gist: Gist):
        if gist.verified:
            self.gists.append(gist)

    def summary(self) -> str:
        if not self.gists:
            return "(empty)"
        return "\n".join([f"[{g.agent}] {g.content}" for g in self.gists[-6:]])

class TaskQueue:
    def __init__(self):
        self.tasks: List[dict] = []

    def add(self, task: dict):
        self.tasks.append(task)

    def claim(self) -> Optional[dict]:
        return self.tasks.pop(0) if self.tasks else None

    def empty(self) -> bool:
        return len(self.tasks) == 0

In [ ]:
def call_llm(system: str, user: str, max_tokens: int = 600, temperature: float = 0.4) -> str:
    response = client.chat_completion(
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        model=MODEL,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()

def verify_gist(gist_content: str, evidence: str) -> bool:
    """Simple admission-time verification."""
    prompt = f"""Is this gist fully supported by the evidence? Reply only YES or NO.

Gist: {gist_content}
Evidence: {evidence[:800]}"""
    ans = call_llm("You are a strict verifier. Answer only YES or NO.", prompt, max_tokens=10, temperature=0.0)
    return "YES" in ans.upper()

In [ ]:
# Shared global structures (DeLM style)
shared_context = SharedContext()
task_queue = TaskQueue()

def agent_step(agent_name: str, role_prompt: str, task: dict) -> Optional[Gist]:
    """One agent claims a task, reads shared context, does work, writes a verified gist."""
    print(f"\n[{agent_name}] claimed: {task.get('type', 'task')}")

    context_summary = shared_context.summary()
    prompt = f"""{role_prompt}

Shared verified context so far:
{context_summary}

Your task:
{json.dumps(task, indent=2)}

Produce a short, factual gist of your result (1-3 sentences). Also include the raw evidence/result."""

    raw = call_llm(
        f"You are the {agent_name} agent. Be precise and grounded.",
        prompt,
        max_tokens=500,
        temperature=0.5,
    )

    # Extract a gist line
    lines = [l.strip() for l in raw.split("\n") if l.strip()]
    gist_text = lines[0] if lines else raw[:200]

    gist = Gist(agent=agent_name, content=gist_text, evidence=raw)
    gist.verified = verify_gist(gist_text, raw)

    if gist.verified:
        shared_context.add(gist)
        print(f"[{agent_name}] verified gist admitted: {gist_text}")
    else:
        print(f"[{agent_name}] gist rejected by verification")

    return gist if gist.verified else None

In [ ]:
# Define the three specialized roles
ANALYZER_ROLE = "You analyze code for bugs, edge cases, complexity, and missing tests. Output concrete findings."
IMPROVER_ROLE = "You improve code based on analysis. Output clean, correct Python improvements."
REVIEWER_ROLE = "You review improved code for correctness and completeness. Score and suggest next actions."

def run_delm_loop(goal: str, starting_code: str, max_rounds: int = 6):
    # Seed the task queue
    task_queue.add({"type": "analyze", "goal": goal, "code": starting_code})

    round_num = 0
    while not task_queue.empty() and round_num < max_rounds:
        round_num += 1
        print(f"\n{'='*50}\nROUND {round_num}\n{'='*50}")

        task = task_queue.claim()
        if not task:
            break

        ttype = task.get("type")
        if ttype == "analyze"::
            agent_step("Analyzer", ANALYZER_ROLE, task)
            # After analysis, queue an improve task
            task_queue.add({"type": "improve", "goal": goal, "code": task.get("code", starting_code)})

        elif ttype == "improve"::
            agent_step("Improver", IMPROVER_ROLE, task)
            # After improve, queue a review
            task_queue.add({"type": "review", "goal": goal})

        elif ttype == "review"::
            gist = agent_step("Reviewer", REVIEWER_ROLE, task)
            # Reviewer can enqueue more work if needed (simple heuristic)
            if gist and "improve" in gist.content.lower():
                task_queue.add({"type": "improve", "goal": goal, "code": starting_code})
            elif gist and "analyze" in gist.content.lower():
                task_queue.add({"type": "analyze", "goal": goal, "code": starting_code})

    print("\n" + "="*50)
    print("FINAL SHARED CONTEXT")
    print("="*50)
    print(shared_context.summary())
    return shared_context

In [ ]:
# Run it
GOAL = "Make the factorial function robust with type hints, error handling for negative numbers, and clear docstring."
CODE = '''
def factorial(n):
    result = 1
    for i in range(1, n+1):
        result *= i
    return result
'''

final_context = run_delm_loop(GOAL, CODE.strip(), max_rounds=5)

## Why this is DeLM-style

- **No central controller** that collects and merges everything.
- Agents **claim** work from a task queue asynchronously (here simulated sequentially for the notebook).
- They read a **shared verified context** of compact gists.
- Only **verified** updates are admitted (admission-time check).
- Agents can enqueue new work based on what they see in the shared context.

This removes the classic bottleneck of “main agent waits for everyone then integrates.”

### Production upgrades
- True async / parallel agents (asyncio or multiple processes)
- Stronger verification (evidence grounding + LLM judge)
- Hierarchical gists (gist → summary → raw) for long contexts
- Tool use so agents can actually run tests / linters
- Persistent shared context (Redis / DB) so agents can join and leave